In [14]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, Literal
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [15]:
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.2)

In [16]:
class QuadState(TypedDict):
    a: float
    b: float
    c: float

    eauation : str  
    discriminant : float
    result : str

In [17]:
def show_equation(state: QuadState):
   equation = f"{state['a']}x^2 + {state['b']}x + {state['c']} = 0"
   return {"eauation": equation}

In [18]:
def calculate_discriminant(state: QuadState) -> QuadState:
  discriminant = state["b"] ** 2 - 4 * state["a"] * state["c"]
  return {"discriminant": discriminant}

In [19]:
def real_root(state: QuadState):
        a, b, c = state["a"], state["b"], state["c"]
        discriminant = state["discriminant"]

        root1 = (-b + discriminant ** 0.5) / (2 * a)
        root2 = (-b - discriminant ** 0.5) / (2 * a)
        result = f"Two real roots: {root1} and {root2}"

        return {"result": result}

In [20]:
def no_real_root(state: QuadState):
 
    result = "No real root exist"
    
    return {"result": result}

In [22]:
def repeated_root(state: QuadState) -> QuadState:
        
        a, b = state["a"], state["b"]
        root = -b / (2 * a)
        result = f"One real root: {root}"
        return {"result": result}

In [23]:
def check_condition(state: QuadState) -> Literal["real_root", "repeated_root", "no_real_root"]:
    if state["discriminant"] > 0:
        return "real_root"
    
    elif state["discriminant"] == 0:
        return "repeated_root"  
    
    else:
        return "no_real_root"

In [27]:
graph = StateGraph(QuadState)

graph.add_node("show_equation",show_equation)
graph.add_node("calculate_discriminant",calculate_discriminant)
graph.add_node("real_root", real_root)
graph.add_node("repeated_root", repeated_root)
graph.add_node("no_real_root", no_real_root)



graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")   

graph.add_conditional_edges("calculate_discriminant", check_condition)
graph.add_edge("real_root", END)
graph.add_edge("repeated_root", END)
graph.add_edge("no_real_root", END)
workflow = graph.compile()


In [29]:
initial_state = {
    "a": 4,
    "b": -5,
    "c": -2
}

workflow.invoke(initial_state) 

{'a': 4,
 'b': -5,
 'c': -2,
 'eauation': '4x^2 + -5x + -2 = 0',
 'discriminant': 57,
 'result': 'Two real roots: 1.5687293044088437 and -0.3187293044088437'}